In [1]:
import os
import pandas as pd
import sqlite3

from contextlib import closing
from pandasql import sqldf
from pathlib import Path

from dsi.dsi import DSI

In [ ]:
help(DSI)

In [2]:
def is_valid_sqlite_with_data(path: str) -> tuple[bool, str]:
    if not os.path.isfile(path):
        return False, "file does not exist"

    try:
        with open(path, "rb") as f:
            header = f.read(16)
        if header != b"SQLite format 3\x00":
            return False, "not a SQLite3 file"
    except OSError as e:
        return False, f"could not read file: {e}"

    try:
        with closing(sqlite3.connect(path)) as conn:
            cur = conn.cursor()

            # Check integrity
            row = cur.execute("PRAGMA integrity_check;").fetchone()
            if not row or row[0].lower() != "ok":
                return False, "SQLite integrity check failed"

            # Find user tables
            tables = cur.execute("""
                SELECT name
                FROM sqlite_master
                WHERE type = 'table'
                  AND name NOT LIKE 'sqlite_%'
                LIMIT 1
            """).fetchall()

            if not tables:
                return False, "valid SQLite file, but no user tables"

            # Check whether any user table has data
            for (table_name,) in cur.execute("""
                SELECT name
                FROM sqlite_master
                WHERE type = 'table'
                  AND name NOT LIKE 'sqlite_%'
            """):
                qname = '"' + table_name.replace('"', '""') + '"'
                count = cur.execute(f"SELECT 1 FROM {qname} LIMIT 1").fetchone()
                if count is not None:
                    return True, "valid SQLite file with data"

            return False, "valid SQLite file, but tables are empty"

    except sqlite3.DatabaseError as e:
        return False, f"SQLite open/query failed: {e}"
    except Exception as e:
        return False, f"unexpected error: {e}"

In [3]:
def is_valid_duckdb_with_data(path: str) -> tuple[bool, str]:
    if not os.path.isfile(path):
        return False, "file does not exist"

    try:
        import duckdb
    except ImportError:
        return False, "duckdb package is not installed"

    # DuckDB does not have a simple fixed header check as convenient as SQLite.
    # The reliable test is: can DuckDB open it and query its catalog?
    try:
        conn = duckdb.connect(path, read_only=True)
        try:
            # Check that catalog is readable by listing tables
            tables = conn.execute("""
                SELECT table_schema, table_name
                FROM information_schema.tables
                WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
                  AND table_type = 'BASE TABLE'
            """).fetchall()

            if not tables:
                return False, "valid DuckDB file, but no user tables"

            # Check whether any table has at least one row
            for schema_name, table_name in tables:
                qschema = '"' + schema_name.replace('"', '""') + '"'
                qtable = '"' + table_name.replace('"', '""') + '"'
                row = conn.execute(
                    f"SELECT 1 FROM {qschema}.{qtable} LIMIT 1"
                ).fetchone()
                if row is not None:
                    return True, "valid DuckDB file with data"

            return False, "valid DuckDB file, but tables are empty"
        finally:
            conn.close()

    except Exception as e:
        return False, f"DuckDB open/query failed: {e}"

In [4]:
def detect_valid_db_with_data(path: str) -> tuple[str | None, bool, str]:
    ok, msg = is_valid_sqlite_with_data(path)
    if ok:
        return "sqlite", True

    ok2, msg2 = is_valid_duckdb_with_data(path)
    if ok2:
        return "duckdb", True

    return None, False

In [5]:
class f_dsi:
    def __init__(self, folder_name:str):
        self.folder_name = folder_name
        folder = Path(folder_name)
        print("folder:", folder)

        print("\n")
        db_path_list = []
        for p in folder.rglob("*"):
            if p.is_file():
                db_path_list.append(p)
                print(p)

        print("\n")
        databases = []
        print(db_path_list)
        for d_id, db_path in enumerate(db_path_list):
            db_info = {}

            print("db_path:",db_path)
            database_name, valid_db = detect_valid_db_with_data(db_path)

            if valid_db:
                _temp = DSI(str(db_path), backend_name=database_name, silence_messages="True")
                db_info['id'] = d_id
                db_info['path'] = str(db_path)
                db_info['name'] = (str(db_path)).split('/')[-1]
                _tbls = _temp.list(True)
                db_info['num_tables'] = len(_tbls)
                db_info['tables'] = _tbls
                _temp.close()
                
                databases.append(db_info)
            else:
                print(f"!!!!Error opening database at {db_path}!!!!")

                # delete the file
                # file_path = Path(db_path)
                # file_path.unlink()

        
        print("\nDatabases:")
        print(databases)
        
        self.df = pd.DataFrame(databases)
        self.df_exp = self.df.explode("tables").rename(columns={"tables": "table"})
        
        
    def f_get_databases(self):
        return self.df
    
    def f_summarize(self, table, db):
        q = f"SELECT path FROM df_exp WHERE \"table\" = '{table}' AND name = '{db}'"
        print(q)
        
        out = sqldf(q, {"df_exp": self.df_exp})
        path_str = out.loc[0, "path"]
        print(path_str)
    
        _temp = DSI(path_str, silence_messages="True")
        l = _temp.summary(collection=True)
        return l


    def f_query(self, table, db, query):
        q = f"SELECT path FROM df_exp WHERE \"table\" = '{table}' AND name = '{db}'"
        print(q)
        
        out = sqldf(q, {"df_exp": self.df_exp})
        path_str = out.loc[0, "path"]
    
        _temp = DSI(path_str, silence_messages="True")
        l = _temp.query(query, collection=True)
        return l

    def f_search(self, table, db, query):
        q = f"SELECT path FROM df_exp WHERE \"table\" = '{table}' AND name = '{db}'"
        print(q)
        
        out = sqldf(q, {"df_exp": self.df_exp})
        path_str = out.loc[0, "path"]
    
        _temp = DSI(path_str, silence_messages="True")
        l = _temp.search(query, collection=True)
        return l

    def f_find(self, table, db, query):
        q = f"SELECT path FROM df_exp WHERE \"table\" = '{table}' AND name = '{db}'"
        print(q)
        
        out = sqldf(q, {"df_exp": self.df_exp})
        path_str = out.loc[0, "path"]
    
        _temp = DSI(path_str, silence_messages="True")
        l = _temp.find(query, collection=True)
        return l

# First pull
```bash
(venv_dsi) pascalgrosset@PN2608148:~/projects/dsi$ python tools/federated/federate_dataset.py tools/federated/input.yaml
Databases will be synchronized to: dsi_databases_1
Created: /home/pascalgrosset/projects/dsi/dsi_databases_1
host_usernames.json does not exist. We will create a new one.

 - Processing database at github:github:https://github.com/lanl/dsi/tree/ai_dev/tools/ai_search/data/oceans_11/ocean_11_datasets.db

 - Processing database at HPC:darwin-fe.lanl.gov:/users/pulido/modelcard2.db
 -- Enter the username for darwin-fe.lanl.gov: pascalgrosset
 -- Could not access the file at darwin-fe.lanl.gov:/users/pulido/modelcard2.db. Skipping this database.

 - Processing database at HPC:ch-fe.lanl.gov:/lustre/scratch5/pascalgrosset/test_db/nif.db
 -- Enter the username for ch-fe.lanl.gov: pascalgrosset
(pascalgrosset@ch-fe.lanl.gov) Password:
(pascalgrosset@ch-fe.lanl.gov) Password:
 -- Interrupted while checking ch-fe.lanl.gov:/lustre/scratch5/pascalgrosset/test_db/nif.db. Skipping this database.

 - Processing database at HPC:darwin-fe.lanl.gov:/vast/projects/exasky/pascal/genesis_model_cards/modelcard.db
      1,093,632 100%    4.38MB/s    0:00:00 (xfr#1, to-chk=0/1)

Finished gathering databases. Successfully downloaded 2 databases to dsi_databases_1.
```


# Second pull
```bash
(venv_dsi) pascalgrosset@PN2608148:~/projects/dsi$ python tools/federated/federate_dataset.py tools/federated/input.yaml
Databases will be synchronized to: dsi_databases_1
Created: /home/pascalgrosset/projects/dsi/dsi_databases_1

 - Processing database at github:github:https://github.com/lanl/dsi/tree/ai_dev/tools/ai_search/data/oceans_11/ocean_11_datasets.db

 - Processing database at HPC:darwin-fe.lanl.gov:/users/pulido/modelcard2.db
 -- Could not access the file at darwin-fe.lanl.gov:/users/pulido/modelcard2.db. Skipping this database.

 - Processing database at HPC:ch-fe.lanl.gov:/lustre/scratch5/pascalgrosset/test_db/nif.db
(pascalgrosset@ch-fe.lanl.gov) Password:
(pascalgrosset@ch-fe.lanl.gov) Password:
 -- Interrupted while checking ch-fe.lanl.gov:/lustre/scratch5/pascalgrosset/test_db/nif.db. Skipping this database.

 - Processing database at HPC:darwin-fe.lanl.gov:/vast/projects/exasky/pascal/genesis_model_cards/modelcard.db
 -- Local file is up to date with the remote file. Skipping download.

 - Processing database at HPC:saul.nersc.gov:/global/u1/p/pascal/duckdb-demo.duckdb
 -- Enter the username for saul.nersc.gov: pascal
(pascal@saul.nersc.gov) Password + OTP:
(pascal@saul.nersc.gov) Password + OTP:
(pascal@saul.nersc.gov) Password + OTP:
***************************************************************************
                          NOTICE TO USERS

Lawrence Berkeley National Laboratory operates this computer system under
contract to the U.S. Department of Energy.  This computer system is the
property of the United States Government and is for authorized use only.
Users (authorized or unauthorized) have no explicit or implicit
expectation of privacy.

Any or all uses of this system and all files on this system may be
intercepted, monitored, recorded, copied, audited, inspected, and disclosed
to authorized site, Department of Energy, and law enforcement personnel,
as well as authorized officials of other agencies, both domestic and foreign.
By using this system, the user consents to such interception, monitoring,
recording, copying, auditing, inspection, and disclosure at the discretion
of authorized site or Department of Energy personnel.

Unauthorized or improper use of this system may result in administrative
disciplinary action and civil and criminal penalties. By continuing to use
this system you indicate your awareness of and consent to these terms and
conditions of use. LOG OFF IMMEDIATELY if you do not agree to the conditions
stated in this warning.

*****************************************************************************

Login connection to host x3116c0s21b0n0:

(pascal@saul.nersc.gov) Password + OTP:
Lmod has detected the following error: The following module(s) are unknown:
"tmux"

Please check the spelling or version number. Also try "module spider ..."
It is also possible your cache file is out-of-date; it may help to try:
  $ module --ignore_cache load "tmux"

Also make sure that all modulefiles written in TCL start with the string
#%Module



      4,993,024 100%    3.37MB/s    0:00:01 (xfr#1, to-chk=0/1)

Finished gathering databases. Successfully downloaded 2 databases to dsi_databases_1.
```

In [6]:
folder_name = "/home/pascalgrosset/projects/dsi/dsi_databases_1/"

In [7]:
x = f_dsi(folder_name)

folder: /home/pascalgrosset/projects/dsi/dsi_databases_1


/home/pascalgrosset/projects/dsi/dsi_databases_1/host_usernames.json
/home/pascalgrosset/projects/dsi/dsi_databases_1/c0b0109d9439de57/ocean_11_datasets.db
/home/pascalgrosset/projects/dsi/dsi_databases_1/fddbb02cd15268a0/duckdb-demo.duckdb
/home/pascalgrosset/projects/dsi/dsi_databases_1/4c2f907a5809ddad/modelcard.db


[PosixPath('/home/pascalgrosset/projects/dsi/dsi_databases_1/host_usernames.json'), PosixPath('/home/pascalgrosset/projects/dsi/dsi_databases_1/c0b0109d9439de57/ocean_11_datasets.db'), PosixPath('/home/pascalgrosset/projects/dsi/dsi_databases_1/fddbb02cd15268a0/duckdb-demo.duckdb'), PosixPath('/home/pascalgrosset/projects/dsi/dsi_databases_1/4c2f907a5809ddad/modelcard.db')]
db_path: /home/pascalgrosset/projects/dsi/dsi_databases_1/host_usernames.json
!!!!Error opening database at /home/pascalgrosset/projects/dsi/dsi_databases_1/host_usernames.json!!!!
db_path: /home/pascalgrosset/projects/dsi/dsi_databases_1/c0b

In [8]:
df = x.f_get_databases()
df

,id,path,name,num_tables,tables
0,1,/home/pascalgrosset/projects/dsi/dsi_databases...,ocean_11_datasets.db,1,[genesis_datacard]
1,2,/home/pascalgrosset/projects/dsi/dsi_databases...,duckdb-demo.duckdb,17,"[bank_failures, boxplot, calendar, candle, chr..."
2,3,/home/pascalgrosset/projects/dsi/dsi_databases...,modelcard.db,1,[models]


In [9]:
 x.f_summarize(table="genesis_datacard", db="ocean_11_datasets.db")

SELECT path FROM df_exp WHERE "table" = 'genesis_datacard' AND name = 'ocean_11_datasets.db'
/home/pascalgrosset/projects/dsi/dsi_databases_1/c0b0109d9439de57/ocean_11_datasets.db


[                                               column     type unique   min  \
 0                                               Title  VARCHAR      9  None   
 1                                        Keywords/Tag  VARCHAR      9  None   
 2                            Update/Modification_Date  VARCHAR      6  None   
 3                             Theme/Category_:_Domain  VARCHAR      5  None   
 4                                        Description_  VARCHAR      9  None   
 5                                      Contact_Point_  VARCHAR      6  None   
 6   Formatted_Name*Format_with_separate_fields_for...  VARCHAR      5  None   
 7                                               Email  VARCHAR      5  None   
 8   Access_Rights_:_SecurityClassification_(Inform...  VARCHAR      1  None   
 9                                     CUI_Restriction  VARCHAR      1  None   
 10                                 CUI_Banner_Marking  VARCHAR      1  None   
 11                           CUI_Design

In [ ]:
#x.f_find(table="genesis_datacard", db="ocean_11_datasets.db", query="dens_max > 5")